In [1]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)

import warnings
warnings.filterwarnings("ignore")

In [2]:
TRAIN_CSV = "/content/drive/MyDrive/assessment/train.csv"
TEST_CSV = "/content/drive/MyDrive/assessment/test.csv"

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nFirst 5 rows:")
display(train_df.head())

Train shape: (769, 2)
Test shape: (216, 2)

Train columns:
['filename', 'label']

First 5 rows:


,filename,label
0,audio_192.wav,3.0
1,audio_436.wav,3.0
2,audio_447.wav,3.5
3,audio_126.wav,2.0
4,audio_61.wav,2.0


In [3]:
print("Missing values:")
print(train_df.isnull().sum())

print("\nLabel statistics:")
print(train_df["label"].describe())

print("\nLabel distribution:")
print(train_df["label"].value_counts().sort_index())

Missing values:
filename    0
label       0
dtype: int64

Label statistics:
count    769.000000
mean       3.313394
std        1.239000
min        0.000000
25%        2.500000
50%        3.000000
75%        4.000000
max        5.000000
Name: label, dtype: float64

Label distribution:
label
0.0     37
1.0      1
1.5      3
2.0    102
2.5     79
3.0    174
3.5     64
4.0    124
4.5     52
5.0    133
Name: count, dtype: int64


In [5]:
import os

AUDIO_DIR = "/content/drive/MyDrive/assessment/train"

print("Folder exists:", os.path.exists(AUDIO_DIR))

if os.path.exists(AUDIO_DIR):
    files = os.listdir(AUDIO_DIR)
    print("Number of files:", len(files))
    print("First 10 files:", files[:10])

Folder exists: True
Number of files: 769
First 10 files: ['audio_0.wav', 'audio_1.wav', 'audio_10.wav', 'audio_100.wav', 'audio_101.wav', 'audio_102.wav', 'audio_103.wav', 'audio_104.wav', 'audio_105.wav', 'audio_106.wav']


In [6]:
def extract_audio_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    features = {}

    # Basic audio features
    features["duration"] = len(y) / sr

    # RMS
    rms = librosa.feature.rms(y=y)[0]
    features["rms_mean"] = np.mean(rms)
    features["rms_std"] = np.std(rms)

    # Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features["zcr_mean"] = np.mean(zcr)
    features["zcr_std"] = np.std(zcr)

    # Spectral features
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features["spectral_centroid_mean"] = np.mean(spectral_centroid)
    features["spectral_centroid_std"] = np.std(spectral_centroid)

    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    features["spectral_bandwidth_mean"] = np.mean(spectral_bandwidth)
    features["spectral_bandwidth_std"] = np.std(spectral_bandwidth)

    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    features["spectral_rolloff_mean"] = np.mean(spectral_rolloff)
    features["spectral_rolloff_std"] = np.std(spectral_rolloff)

    # MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    for i in range(13):
        features[f"mfcc_{i+1}_mean"] = np.mean(mfcc[i])
        features[f"mfcc_{i+1}_std"] = np.std(mfcc[i])

    return features

In [7]:
sample_file = os.path.join(AUDIO_DIR, train_df.iloc[0]["filename"])

sample_features = extract_audio_features(sample_file)

print("Number of features:", len(sample_features))
print(sample_features)

Number of features: 37
{'duration': 60.508375, 'rms_mean': np.float32(0.031785965), 'rms_std': np.float32(0.038403988), 'zcr_mean': np.float64(0.08476779399127446), 'zcr_std': np.float64(0.053518039289684), 'spectral_centroid_mean': np.float64(979.478539311199), 'spectral_centroid_std': np.float64(399.5678879100371), 'spectral_bandwidth_mean': np.float64(916.4720665613124), 'spectral_bandwidth_std': np.float64(261.620566146588), 'spectral_rolloff_mean': np.float64(1770.7396879957694), 'spectral_rolloff_std': np.float64(661.6676618284793), 'mfcc_1_mean': np.float32(-403.8359), 'mfcc_1_std': np.float32(121.62919), 'mfcc_2_mean': np.float32(103.57634), 'mfcc_2_std': np.float32(68.03565), 'mfcc_3_mean': np.float32(-14.213533), 'mfcc_3_std': np.float32(37.100353), 'mfcc_4_mean': np.float32(16.673616), 'mfcc_4_std': np.float32(21.843693), 'mfcc_5_mean': np.float32(7.04855), 'mfcc_5_std': np.float32(11.192187), 'mfcc_6_mean': np.float32(-4.9996395), 'mfcc_6_std': np.float32(14.073897), 'mfcc_

In [8]:
from tqdm.auto import tqdm

feature_rows = []

for filename in tqdm(train_df["filename"], desc="Extracting audio features"):
    file_path = os.path.join(AUDIO_DIR, filename)

    features = extract_audio_features(file_path)
    features["filename"] = filename

    feature_rows.append(features)

features_df = pd.DataFrame(feature_rows)

print("Feature dataframe shape:", features_df.shape)
display(features_df.head())

Extracting audio features:   0%|          | 0/769 [00:00<?, ?it/s]

Feature dataframe shape: (769, 38)


,duration,rms_mean,rms_std,zcr_mean,zcr_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,...,mfcc_9_std,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,filename
0,60.508375,0.031786,0.038404,0.084768,0.053518,979.478539,399.567888,916.472067,261.620566,1770.739688,...,9.175092,-6.111350,7.005802,-11.947043,8.755229,-8.226156,6.883492,-6.238205,5.747041,audio_192.wav
1,60.074688,0.068651,0.055423,0.142977,0.127519,1813.837012,1156.460316,1502.301143,495.435560,3277.169030,...,16.002325,-8.470199,13.448154,-9.261223,11.554407,-9.040758,13.295260,4.751498,13.532232,audio_436.wav
2,60.074688,0.054421,0.051533,0.146870,0.129777,1652.475254,1207.838332,1500.652559,535.214385,2941.260650,...,16.530882,-8.570185,16.668097,3.306296,13.158398,-3.113312,11.855078,-0.866495,11.763124,audio_447.wav
3,45.060000,0.013778,0.012183,0.059477,0.036870,996.631363,471.566967,1177.252185,392.434373,1865.640525,...,15.149113,-21.515909,15.097207,-12.347360,11.821059,-9.993058,10.599061,-17.710949,11.394984,audio_126.wav
4,45.300000,0.009330,0.008970,0.169988,0.128430,2278.144095,1068.412091,1630.415181,442.415960,4034.356241,...,16.179859,-18.795961,16.087769,-8.590364,13.599857,3.873606,10.692766,-17.764479,15.302751,audio_61.wav


In [9]:
data = features_df.merge(
    train_df,
    on="filename",
    how="inner"
)

print("Final shape:", data.shape)

print("\nMissing values:")
print(data.isnull().sum().sum())

display(data.head())

Final shape: (769, 39)

Missing values:
0


,duration,rms_mean,rms_std,zcr_mean,zcr_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,...,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,filename,label
0,60.508375,0.031786,0.038404,0.084768,0.053518,979.478539,399.567888,916.472067,261.620566,1770.739688,...,-6.111350,7.005802,-11.947043,8.755229,-8.226156,6.883492,-6.238205,5.747041,audio_192.wav,3.0
1,60.074688,0.068651,0.055423,0.142977,0.127519,1813.837012,1156.460316,1502.301143,495.435560,3277.169030,...,-8.470199,13.448154,-9.261223,11.554407,-9.040758,13.295260,4.751498,13.532232,audio_436.wav,3.0
2,60.074688,0.054421,0.051533,0.146870,0.129777,1652.475254,1207.838332,1500.652559,535.214385,2941.260650,...,-8.570185,16.668097,3.306296,13.158398,-3.113312,11.855078,-0.866495,11.763124,audio_447.wav,3.5
3,45.060000,0.013778,0.012183,0.059477,0.036870,996.631363,471.566967,1177.252185,392.434373,1865.640525,...,-21.515909,15.097207,-12.347360,11.821059,-9.993058,10.599061,-17.710949,11.394984,audio_126.wav,2.0
4,45.300000,0.009330,0.008970,0.169988,0.128430,2278.144095,1068.412091,1630.415181,442.415960,4034.356241,...,-18.795961,16.087769,-8.590364,13.599857,3.873606,10.692766,-17.764479,15.302751,audio_61.wav,2.0


In [10]:
feature_cols = [
    col for col in data.columns
    if col not in ["filename", "label"]
]

X = data[feature_cols]
y = data["label"]

print("Number of features:", X.shape[1])
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 37
X shape: (769, 37)
y shape: (769,)


In [11]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)

Training: (615, 37)
Validation: (154, 37)


In [12]:
from sklearn.ensemble import ExtraTreesRegressor

extra_model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1,
    max_features=1.0,
    min_samples_leaf=2
)

extra_model.fit(X_train, y_train)

pred_extra = extra_model.predict(X_val)

rmse_extra = np.sqrt(mean_squared_error(y_val, pred_extra))
pearson_extra = pearsonr(y_val, pred_extra)[0]

print("ExtraTrees Results")
print("------------------")
print("RMSE:", rmse_extra)
print("Pearson:", pearson_extra)

ExtraTrees Results
------------------
RMSE: 0.6842597045376618
Pearson: 0.8650801148950545


In [13]:
!pip install -q xgboost


In [14]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

pred_xgb = xgb_model.predict(X_val)

rmse_xgb = np.sqrt(mean_squared_error(y_val, pred_xgb))
pearson_xgb = pearsonr(y_val, pred_xgb)[0]

print("XGBoost Results")
print("----------------")
print("RMSE:", rmse_xgb)
print("Pearson:", pearson_xgb)

XGBoost Results
----------------
RMSE: 0.7152716031152969
Pearson: 0.8494006536107414


In [15]:
from sklearn.ensemble import ExtraTreesRegressor

configs = [
    {
        "n_estimators": 500,
        "max_features": 1.0,
        "min_samples_leaf": 1,
        "max_depth": None
    },
    {
        "n_estimators": 500,
        "max_features": 0.8,
        "min_samples_leaf": 1,
        "max_depth": None
    },
    {
        "n_estimators": 500,
        "max_features": 0.7,
        "min_samples_leaf": 2,
        "max_depth": None
    },
    {
        "n_estimators": 500,
        "max_features": 0.8,
        "min_samples_leaf": 2,
        "max_depth": None
    },
    {
        "n_estimators": 500,
        "max_features": 0.7,
        "min_samples_leaf": 3,
        "max_depth": None
    },
    {
        "n_estimators": 500,
        "max_features": 1.0,
        "min_samples_leaf": 2,
        "max_depth": 15
    },
    {
        "n_estimators": 500,
        "max_features": 0.8,
        "min_samples_leaf": 2,
        "max_depth": 15
    },
    {
        "n_estimators": 500,
        "max_features": 0.8,
        "min_samples_leaf": 3,
        "max_depth": 15
    }
]

results = []

for i, params in enumerate(configs, start=1):

    model = ExtraTreesRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    pearson = pearsonr(y_val, pred)[0]

    results.append({
        "config": i,
        "RMSE": rmse,
        "Pearson": pearson,
        **params
    })

    print(
        f"Config {i}: "
        f"RMSE={rmse:.4f}, "
        f"Pearson={pearson:.4f}"
    )

Config 1: RMSE=0.6843, Pearson=0.8648
Config 2: RMSE=0.6833, Pearson=0.8656
Config 3: RMSE=0.6883, Pearson=0.8638
Config 4: RMSE=0.6886, Pearson=0.8633
Config 5: RMSE=0.6990, Pearson=0.8591
Config 6: RMSE=0.6866, Pearson=0.8639
Config 7: RMSE=0.6889, Pearson=0.8633
Config 8: RMSE=0.7033, Pearson=0.8568


In [16]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

display(results_df)

,config,RMSE,Pearson,n_estimators,max_features,min_samples_leaf,max_depth
0,2,0.683310,0.865608,500,0.8,1,NaN
1,1,0.684313,0.864840,500,1.0,1,NaN
2,6,0.686596,0.863895,500,1.0,2,15.0
3,3,0.688268,0.863813,500,0.7,2,NaN
4,4,0.688623,0.863253,500,0.8,2,NaN
5,7,0.688936,0.863342,500,0.8,2,15.0
6,5,0.699041,0.859082,500,0.7,3,NaN
7,8,0.703314,0.856776,500,0.8,3,15.0


In [17]:
import os

print("Files/folders in /content:\n")

for item in os.listdir("/content"):
    print(item)

Files/folders in /content:

.config
drive
sample_data


In [18]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        file_lower = file.lower()

        if (
            "transcript" in file_lower
            or "text" in file_lower
            or file_lower.endswith(".csv")
            or file_lower.endswith(".json")
            or file_lower.endswith(".txt")
        ):
            print(os.path.join(root, file))

/content/drive/MyDrive/Java_Backend_Study_Plan.csv
/content/drive/MyDrive/Colab Notebooks/DataPreProcessing.csv
/content/drive/MyDrive/Google AI Studio/applet_access_history.json
/content/drive/MyDrive/assessment/sample_submission.csv
/content/drive/MyDrive/assessment/test.csv
/content/drive/MyDrive/assessment/train.csv
/content/drive/MyDrive/assessment/train_transcripts.csv
/content/drive/MyDrive/assessment/submission_v1_audio_rf.csv


In [19]:
TRANSCRIPT_CSV = "/content/drive/MyDrive/assessment/train_transcripts.csv"

transcripts_df = pd.read_csv(TRANSCRIPT_CSV)

print("Shape:", transcripts_df.shape)
print("\nColumns:")
print(transcripts_df.columns.tolist())

print("\nFirst 5 rows:")
display(transcripts_df.head())

print("\nMissing values:")
print(transcripts_df.isnull().sum())

Shape: (28, 2)

Columns:
['filename', 'transcript']

First 5 rows:


,filename,transcript
0,audio_192.wav,"Well, this petite land is very, very beautiful..."
1,audio_436.wav,"To see no of a school playground, our playgrou..."
2,audio_447.wav,I favorite place to visit is the beach. I love...
3,audio_126.wav,"Once upon a time, we, with five friends, went ..."
4,audio_61.wav,"Well, when I was a child, I remember I went to..."



Missing values:
filename      0
transcript    0
dtype: int64


In [20]:
transcript_data = transcripts_df.merge(
    train_df,
    on="filename",
    how="inner"
)

print("Shape:", transcript_data.shape)

display(
    transcript_data[
        ["filename", "transcript", "label"]
    ].head(10)
)

Shape: (28, 3)


,filename,transcript,label
0,audio_192.wav,"Well, this petite land is very, very beautiful...",3.0
1,audio_436.wav,"To see no of a school playground, our playgrou...",3.0
2,audio_447.wav,I favorite place to visit is the beach. I love...,3.5
3,audio_126.wav,"Once upon a time, we, with five friends, went ...",2.0
4,audio_61.wav,"Well, when I was a child, I remember I went to...",2.0
5,audio_639.wav,The best thing of my life was probably when I ...,2.5
6,audio_703.wav,I don't know. Do you want to talk to me? I nev...,2.0
7,audio_509.wav,is smoothing effects. Besides this I also have...,3.0
8,audio_672.wav,The school playground is very big and is very ...,4.0
9,audio_344.wav,My normal goal in life is to become the best o...,4.0


In [21]:
print("Label statistics for transcript samples:")
print(transcript_data["label"].describe())

print("\nTranscript lengths:")
transcript_data["word_count"] = (
    transcript_data["transcript"]
    .str.split()
    .str.len()
)

print(transcript_data["word_count"].describe())

Label statistics for transcript samples:
count    28.000000
mean      3.178571
std       1.107335
min       0.000000
25%       2.375000
50%       3.000000
75%       4.000000
max       5.000000
Name: label, dtype: float64

Transcript lengths:
count     28.000000
mean      98.785714
std       24.730184
min       50.000000
25%       79.750000
50%       99.000000
75%      113.250000
max      150.000000
Name: word_count, dtype: float64


In [22]:
import re

def extract_text_features(text):
    words = re.findall(r"\b[a-zA-Z']+\b", text.lower())

    if not words:
        return {
            "word_count": 0,
            "unique_word_count": 0,
            "unique_word_ratio": 0,
            "avg_word_length": 0,
            "long_word_ratio": 0,
            "short_word_ratio": 0,
            "filler_count": 0,
            "filler_ratio": 0,
            "repeated_word_ratio": 0
        }

    word_count = len(words)
    unique_word_count = len(set(words))

    # Vocabulary richness
    unique_word_ratio = unique_word_count / word_count

    # Average word length
    avg_word_length = np.mean([len(w) for w in words])

    # Long / short words
    long_words = sum(len(w) >= 7 for w in words)
    short_words = sum(len(w) <= 2 for w in words)

    long_word_ratio = long_words / word_count
    short_word_ratio = short_words / word_count

    # Common speech fillers
    fillers = {
        "um", "uh", "erm", "hmm", "like",
        "you know", "actually", "basically"
    }

    filler_count = sum(
        1 for w in words if w in fillers
    )

    filler_ratio = filler_count / word_count

    # Repeated consecutive words
    repeated_words = sum(
        words[i] == words[i-1]
        for i in range(1, len(words))
    )

    repeated_word_ratio = repeated_words / word_count

    return {
        "word_count": word_count,
        "unique_word_count": unique_word_count,
        "unique_word_ratio": unique_word_ratio,
        "avg_word_length": avg_word_length,
        "long_word_ratio": long_word_ratio,
        "short_word_ratio": short_word_ratio,
        "filler_count": filler_count,
        "filler_ratio": filler_ratio,
        "repeated_word_ratio": repeated_word_ratio
    }

In [23]:
text_features = transcript_data["transcript"].apply(
    extract_text_features
)

text_features_df = pd.DataFrame(
    text_features.tolist()
)

transcript_features = pd.concat(
    [
        transcript_data[["filename", "label"]],
        text_features_df
    ],
    axis=1
)

print("Shape:", transcript_features.shape)

display(transcript_features.head())

Shape: (28, 11)


,filename,label,word_count,unique_word_count,unique_word_ratio,avg_word_length,long_word_ratio,short_word_ratio,filler_count,filler_ratio,repeated_word_ratio
0,audio_192.wav,3.0,91,62,0.681319,4.087912,0.087912,0.175824,4,0.043956,0.010989
1,audio_436.wav,3.0,116,55,0.474138,4.258621,0.146552,0.250000,6,0.051724,0.008621
2,audio_447.wav,3.5,150,74,0.493333,3.846667,0.106667,0.286667,1,0.006667,0.000000
3,audio_126.wav,2.0,99,45,0.454545,3.616162,0.040404,0.323232,0,0.000000,0.020202
4,audio_61.wav,2.0,74,44,0.594595,3.810811,0.081081,0.202703,0,0.000000,0.013514


In [24]:
correlation = transcript_features.drop(
    columns=["filename", "label"]
).corrwith(
    transcript_features["label"]
).sort_values(
    key=abs,
    ascending=False
)

print("Correlation with grammar score:")
display(correlation)

Correlation with grammar score:


,0
unique_word_count,0.443915
word_count,0.408964
repeated_word_ratio,-0.243423
unique_word_ratio,-0.196621
long_word_ratio,0.167149
short_word_ratio,0.099034
avg_word_length,0.045866
filler_count,-0.002224
filler_ratio,-0.000728


In [25]:
def extract_speech_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    features = {}

    # --------------------------------------------------
    # 1. Pitch / Fundamental Frequency
    # --------------------------------------------------
    f0, voiced_flag, voiced_prob = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr
    )

    valid_f0 = f0[~np.isnan(f0)]

    if len(valid_f0) > 0:
        features["f0_mean"] = np.mean(valid_f0)
        features["f0_std"] = np.std(valid_f0)
        features["f0_min"] = np.min(valid_f0)
        features["f0_max"] = np.max(valid_f0)
        features["f0_range"] = np.max(valid_f0) - np.min(valid_f0)
    else:
        features["f0_mean"] = 0
        features["f0_std"] = 0
        features["f0_min"] = 0
        features["f0_max"] = 0
        features["f0_range"] = 0

    # Voiced ratio
    features["voiced_ratio"] = np.mean(voiced_flag)

    # --------------------------------------------------
    # 2. Silence / Energy features
    # --------------------------------------------------
    rms = librosa.feature.rms(y=y)[0]

    # Threshold based on RMS
    threshold = np.mean(rms) * 0.1

    silence_frames = rms < threshold

    features["silence_ratio"] = np.mean(silence_frames)
    features["active_ratio"] = 1 - features["silence_ratio"]

    # --------------------------------------------------
    # 3. Energy dynamics
    # --------------------------------------------------
    features["energy_mean"] = np.mean(rms)
    features["energy_std"] = np.std(rms)
    features["energy_max"] = np.max(rms)

    # Energy variation
    if len(rms) > 1:
        features["energy_change_mean"] = np.mean(
            np.abs(np.diff(rms))
        )
    else:
        features["energy_change_mean"] = 0

    # --------------------------------------------------
    # 4. Spectral contrast
    # --------------------------------------------------
    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr
    )

    features["spectral_contrast_mean"] = np.mean(contrast)
    features["spectral_contrast_std"] = np.std(contrast)

    # --------------------------------------------------
    # 5. Spectral flatness
    # --------------------------------------------------
    flatness = librosa.feature.spectral_flatness(y=y)[0]

    features["spectral_flatness_mean"] = np.mean(flatness)
    features["spectral_flatness_std"] = np.std(flatness)

    return features

In [26]:
sample_file = os.path.join(
    AUDIO_DIR,
    train_df.iloc[0]["filename"]
)

speech_test = extract_speech_features(sample_file)

print("Number of new speech features:", len(speech_test))
print(speech_test)

Number of new speech features: 16
{'f0_mean': np.float64(166.77212864519328), 'f0_std': np.float64(22.696892129645587), 'f0_min': np.float64(65.40639132514966), 'f0_max': np.float64(244.1052838549259), 'f0_range': np.float64(178.69889252977623), 'voiced_ratio': np.float64(0.6070861977789529), 'silence_ratio': np.float64(0.39767318878900054), 'active_ratio': np.float64(0.6023268112109994), 'energy_mean': np.float32(0.031785965), 'energy_std': np.float32(0.038403988), 'energy_max': np.float32(0.18884182), 'energy_change_mean': np.float32(0.006581384), 'spectral_contrast_mean': np.float64(19.250808392826727), 'spectral_contrast_std': np.float64(7.21812993188449), 'spectral_flatness_mean': np.float32(0.0025811652), 'spectral_flatness_std': np.float32(0.00401799)}


In [28]:
def extract_fast_speech_features(file_path):
    y, sr = librosa.load(file_path, sr=16000)

    features = {}

    # -----------------------------
    # 1. RMS / Energy
    # -----------------------------
    rms = librosa.feature.rms(y=y)[0]

    features["energy_mean"] = np.mean(rms)
    features["energy_std"] = np.std(rms)
    features["energy_max"] = np.max(rms)

    if len(rms) > 1:
        features["energy_change_mean"] = np.mean(
            np.abs(np.diff(rms))
        )
    else:
        features["energy_change_mean"] = 0

    # -----------------------------
    # 2. Silence / Activity
    # -----------------------------
    threshold = np.mean(rms) * 0.1

    silence = rms < threshold

    features["silence_ratio"] = np.mean(silence)
    features["active_ratio"] = 1 - features["silence_ratio"]

    # -----------------------------
    # 3. Zero Crossing Rate
    # -----------------------------
    zcr = librosa.feature.zero_crossing_rate(y)[0]

    features["zcr_mean_fast"] = np.mean(zcr)
    features["zcr_std_fast"] = np.std(zcr)
    features["zcr_max_fast"] = np.max(zcr)

    # -----------------------------
    # 4. Spectral Contrast
    # -----------------------------
    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr
    )

    features["spectral_contrast_mean"] = np.mean(contrast)
    features["spectral_contrast_std"] = np.std(contrast)

    # -----------------------------
    # 5. Spectral Flatness
    # -----------------------------
    flatness = librosa.feature.spectral_flatness(y=y)[0]

    features["spectral_flatness_mean"] = np.mean(flatness)
    features["spectral_flatness_std"] = np.std(flatness)

    return features

In [30]:
sample_file = os.path.join(
    AUDIO_DIR,
    train_df.iloc[0]["filename"]
)

fast_test = extract_fast_speech_features(sample_file)

print("Number of fast speech features:", len(fast_test))
print(fast_test)

Number of fast speech features: 13
{'energy_mean': np.float32(0.031785965), 'energy_std': np.float32(0.038403988), 'energy_max': np.float32(0.18884182), 'energy_change_mean': np.float32(0.006581384), 'silence_ratio': np.float64(0.39767318878900054), 'active_ratio': np.float64(0.6023268112109994), 'zcr_mean_fast': np.float64(0.08476779399127446), 'zcr_std_fast': np.float64(0.053518039289684), 'zcr_max_fast': np.float64(0.3359375), 'spectral_contrast_mean': np.float64(19.250808392826727), 'spectral_contrast_std': np.float64(7.21812993188449), 'spectral_flatness_mean': np.float32(0.0025811652), 'spectral_flatness_std': np.float32(0.00401799)}


In [32]:
TEST_AUDIO_DIR = "/content/drive/MyDrive/assessment/test"

print("Test folder exists:", os.path.exists(TEST_AUDIO_DIR))

if os.path.exists(TEST_AUDIO_DIR):
    test_audio_files = os.listdir(TEST_AUDIO_DIR)
    print("Number of test audio files:", len(test_audio_files))
    print("First 10:", test_audio_files[:10])

Test folder exists: True
Number of test audio files: 216
First 10: ['audio_0.wav', 'audio_1.wav', 'audio_10.wav', 'audio_100.wav', 'audio_101.wav', 'audio_102.wav', 'audio_103.wav', 'audio_104.wav', 'audio_105.wav', 'audio_106.wav']


In [33]:
test_feature_rows = []

for filename in tqdm(
    test_df["filename"],
    desc="Extracting test audio features"
):
    file_path = os.path.join(TEST_AUDIO_DIR, filename)

    features = extract_audio_features(file_path)
    features["filename"] = filename

    test_feature_rows.append(features)

test_features_df = pd.DataFrame(test_feature_rows)

print("Test feature shape:", test_features_df.shape)
display(test_features_df.head())

Extracting test audio features:   0%|          | 0/216 [00:00<?, ?it/s]

Test feature shape: (216, 38)


,duration,rms_mean,rms_std,zcr_mean,zcr_std,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,...,mfcc_9_std,mfcc_10_mean,mfcc_10_std,mfcc_11_mean,mfcc_11_std,mfcc_12_mean,mfcc_12_std,mfcc_13_mean,mfcc_13_std,filename
0,45.060000,0.018476,0.018033,0.091145,0.061664,1313.264772,629.115488,1370.977849,523.550599,2646.685371,...,10.750962,-8.662834,11.958874,-2.093965,8.446618,-2.751901,8.443857,-5.733756,8.805126,audio_128.wav
1,44.969313,0.046379,0.045599,0.186478,0.134429,1887.407841,1131.228708,1602.602286,499.222545,3499.322102,...,17.222078,-18.275993,13.206761,-6.153337,13.933970,-5.587032,10.239784,-7.673637,9.794288,audio_156.wav
2,45.012000,0.048284,0.044188,0.162025,0.082149,1677.448618,724.928917,1313.941010,388.693842,2978.244936,...,13.497080,-10.828346,12.366035,-17.725065,11.668586,-2.528805,10.497900,-7.882829,9.199354,audio_19.wav
3,58.980000,0.005259,0.005214,0.159751,0.109905,1915.431594,861.140927,1736.017440,322.650005,3641.807043,...,9.221682,-1.623224,10.203712,-9.178816,7.352479,-6.522888,6.525156,-6.726883,7.192762,audio_108.wav
4,45.040000,0.058809,0.038030,0.064533,0.065902,1171.666526,725.126069,1447.879524,452.048982,2234.746760,...,14.190731,-0.635752,11.704207,4.300982,9.297211,-1.969271,11.009748,1.723401,9.557956,audio_206.wav


In [34]:
feature_cols = [
    col for col in features_df.columns
    if col != "filename"
]

X_final = features_df[feature_cols]
y_final = train_df["label"]

X_test_final = test_features_df[feature_cols]

print("Training:", X_final.shape)
print("Test:", X_test_final.shape)

Training: (769, 37)
Test: (216, 37)


In [35]:
final_model = ExtraTreesRegressor(
    n_estimators=500,
    max_features=0.8,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_final, y_final)

test_predictions = final_model.predict(X_test_final)

print("First 10 predictions:")
print(np.round(test_predictions[:10], 3))

print("\nMin prediction:", test_predictions.min())
print("Max prediction:", test_predictions.max())

First 10 predictions:
[3.215 3.945 3.368 2.609 3.293 2.906 2.982 3.061 3.401 3.315]

Min prediction: 2.494
Max prediction: 4.244


In [36]:
test_predictions = np.clip(
    test_predictions,
    0,
    5
)

print("After clipping:")
print("Min:", test_predictions.min())
print("Max:", test_predictions.max())

After clipping:
Min: 2.494
Max: 4.244


In [37]:
submission = pd.DataFrame({
    "filename": test_df["filename"],
    "label": test_predictions
})

print("Rows:", len(submission))
print("Unique filenames:", submission["filename"].nunique())
print("\nMissing values:")
print(submission.isnull().sum())

display(submission.head(10))

Rows: 216
Unique filenames: 216

Missing values:
filename    0
label       0
dtype: int64


,filename,label
0,audio_128.wav,3.215
1,audio_156.wav,3.945
2,audio_19.wav,3.368
3,audio_108.wav,2.609
4,audio_206.wav,3.293
5,audio_161.wav,2.906
6,audio_215.wav,2.982
7,audio_213.wav,3.061
8,audio_11.wav,3.401
9,audio_155.wav,3.315


In [38]:
submission_path = "/content/drive/MyDrive/assessment/submission_extratrees_v2.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("Saved:", submission_path)

Saved: /content/drive/MyDrive/assessment/submission_extratrees_v2.csv
